# 01: Exploración del Dataset SIRCCD

Este notebook explora el dataset de detección de daños viales después del procesamiento y antes del entrenamiento.

**Objetivos**:
1. Verificar estructura del dataset
2. Analizar distribución de clases
3. Visualizar imágenes y anotaciones
4. Calcular estadísticas del dataset
5. Verificar data augmentation

## 1. Setup y Configuración

In [ ]:
# Imports
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from collections import Counter, defaultdict
from dotenv import load_dotenv
import json
from tqdm.notebook import tqdm

# Cargar variables de entorno
load_dotenv('../.env')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Librerías importadas")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [ ]:
# Rutas
DATASET_DIR = Path('../datasets/processed/split')
METADATA_DIR = Path('../datasets/metadata')

TRAIN_IMAGES = DATASET_DIR / 'images' / 'train'
TRAIN_LABELS = DATASET_DIR / 'labels' / 'train'
VAL_IMAGES = DATASET_DIR / 'images' / 'val'
VAL_LABELS = DATASET_DIR / 'labels' / 'val'
TEST_IMAGES = DATASET_DIR / 'images' / 'test'
TEST_LABELS = DATASET_DIR / 'labels' / 'test'

# Verificar existencia
for path in [TRAIN_IMAGES, TRAIN_LABELS, VAL_IMAGES, VAL_LABELS, TEST_IMAGES, TEST_LABELS]:
    if path.exists():
        print(f"✅ {path}")
    else:
        print(f"❌ {path} - NO EXISTE")

## 2. Estadísticas del Dataset

In [ ]:
# Contar imágenes por split
def count_images(dir_path):
    return len(list(dir_path.glob('*.jpg'))) + len(list(dir_path.glob('*.png')))

train_count = count_images(TRAIN_IMAGES)
val_count = count_images(VAL_IMAGES)
test_count = count_images(TEST_IMAGES)
total_count = train_count + val_count + test_count

print("📊 Distribución de Splits:")
print(f"   Train: {train_count:,} ({train_count/total_count*100:.1f}%)")
print(f"   Val:   {val_count:,} ({val_count/total_count*100:.1f}%)")
print(f"   Test:  {test_count:,} ({test_count/total_count*100:.1f}%)")
print(f"   TOTAL: {total_count:,}")

In [ ]:
# Analizar distribución de clases
def analyze_labels(label_dir):
    """Cuenta objetos por clase en un directorio de labels."""
    class_counts = Counter()
    image_class_counts = Counter()  # Imágenes por clase principal
    
    for label_file in label_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            lines = f.readlines()
            
        if not lines:
            continue
            
        # Contar objetos
        for line in lines:
            parts = line.strip().split()
            if parts:
                class_id = int(parts[0])
                class_counts[class_id] += 1
        
        # Contar imagen por su clase principal (primera anotación)
        main_class = int(lines[0].strip().split()[0])
        image_class_counts[main_class] += 1
    
    return class_counts, image_class_counts

train_objs, train_imgs = analyze_labels(TRAIN_LABELS)
val_objs, val_imgs = analyze_labels(VAL_LABELS)
test_objs, test_imgs = analyze_labels(TEST_LABELS)

# Nombres de clases
class_names = {0: 'bache', 1: 'grieta'}

print("\n📊 Distribución de Objetos por Clase:")
print("\nTRAIN:")
for class_id, count in sorted(train_objs.items()):
    print(f"   {class_names.get(class_id, f'clase_{class_id}')}: {count:,} objetos ({count/sum(train_objs.values())*100:.1f}%)")

print("\nVAL:")
for class_id, count in sorted(val_objs.items()):
    print(f"   {class_names.get(class_id, f'clase_{class_id}')}: {count:,} objetos ({count/sum(val_objs.values())*100:.1f}%)")

print("\nTEST:")
for class_id, count in sorted(test_objs.items()):
    print(f"   {class_names.get(class_id, f'clase_{class_id}')}: {count:,} objetos ({count/sum(test_objs.values())*100:.1f}%)")

## 3. Visualización de Distribuciones

In [ ]:
# Gráfico de barras
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de splits
splits = ['Train', 'Val', 'Test']
counts = [train_count, val_count, test_count]
axes[0].bar(splits, counts, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Imágenes por Split', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de Imágenes')
axes[0].grid(axis='y', alpha=0.3)
for i, count in enumerate(counts):
    axes[0].text(i, count + 500, f'{count:,}', ha='center', fontweight='bold')

# Distribución de clases en train
class_labels = [class_names[i] for i in sorted(train_objs.keys())]
class_values = [train_objs[i] for i in sorted(train_objs.keys())]
axes[1].bar(class_labels, class_values, color=['#9b59b6', '#f39c12'])
axes[1].set_title('Distribución de Objetos por Clase (Train)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de Objetos')
axes[1].grid(axis='y', alpha=0.3)
for i, count in enumerate(class_values):
    axes[1].text(i, count + 200, f'{count:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Visualización de Imágenes

In [ ]:
def draw_yolo_boxes(image_path, label_path, class_names):
    """Dibuja bounding boxes YOLO en una imagen."""
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Leer anotaciones
    if not label_path.exists():
        return img
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    # Colores por clase
    colors = {
        0: (155, 89, 182),  # bache - morado
        1: (243, 156, 18),  # grieta - naranja
    }
    
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        
        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:5])
        
        # Convertir de YOLO a coordenadas de píxeles
        x1 = int((x_center - width / 2) * w)
        y1 = int((y_center - height / 2) * h)
        x2 = int((x_center + width / 2) * w)
        y2 = int((y_center + height / 2) * h)
        
        # Dibujar bbox
        color = colors.get(class_id, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        # Etiqueta
        label = class_names.get(class_id, f'clase_{class_id}')
        cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                   0.5, color, 2)
    
    return img

# Visualizar muestras aleatorias
np.random.seed(42)
train_images = list(TRAIN_IMAGES.glob('*.jpg'))
sample_images = np.random.choice(train_images, min(9, len(train_images)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    label_path = TRAIN_LABELS / f"{img_path.stem}.txt"
    img_with_boxes = draw_yolo_boxes(img_path, label_path, class_names)
    
    axes[i].imshow(img_with_boxes)
    axes[i].set_title(img_path.name, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Muestras Aleatorias del Dataset (Train)', fontsize=16, fontweight='bold', y=1.00)
plt.show()

## 5. Análisis de Tamaños de Bounding Boxes

In [ ]:
def analyze_bbox_sizes(label_dir):
    """Analiza tamaños de bounding boxes."""
    bbox_data = defaultdict(list)
    
    for label_file in tqdm(list(label_dir.glob('*.txt')), desc="Analizando bboxes"):
        with open(label_file, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:5])
            area = width * height  # Área relativa
            
            bbox_data[class_id].append({
                'width': width,
                'height': height,
                'area': area,
                'aspect_ratio': width / height if height > 0 else 0
            })
    
    return bbox_data

bbox_data = analyze_bbox_sizes(TRAIN_LABELS)

# Visualizar distribuciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for class_id, data in bbox_data.items():
    df = pd.DataFrame(data)
    class_label = class_names.get(class_id, f'clase_{class_id}')
    
    # Área
    axes[0, 0].hist(df['area'], bins=50, alpha=0.6, label=class_label)
    axes[0, 0].set_title('Distribución de Áreas')
    axes[0, 0].set_xlabel('Área Relativa')
    axes[0, 0].set_ylabel('Frecuencia')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Aspect ratio
    axes[0, 1].hist(df['aspect_ratio'], bins=50, alpha=0.6, label=class_label)
    axes[0, 1].set_title('Distribución de Aspect Ratios')
    axes[0, 1].set_xlabel('Ancho / Alto')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # Width vs Height scatter
    axes[1, 0].scatter(df['width'], df['height'], alpha=0.3, label=class_label, s=10)
    axes[1, 0].set_title('Ancho vs Alto')
    axes[1, 0].set_xlabel('Ancho Relativo')
    axes[1, 0].set_ylabel('Alto Relativo')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

# Estadísticas
stats_text = "Estadísticas de Bounding Boxes:\n\n"
for class_id, data in bbox_data.items():
    df = pd.DataFrame(data)
    class_label = class_names.get(class_id, f'clase_{class_id}')
    stats_text += f"{class_label}:\n"
    stats_text += f"  Área media: {df['area'].mean():.4f}\n"
    stats_text += f"  Área std: {df['area'].std():.4f}\n"
    stats_text += f"  AR medio: {df['aspect_ratio'].mean():.2f}\n\n"

axes[1, 1].text(0.1, 0.5, stats_text, fontsize=10, verticalalignment='center',
               family='monospace')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## 6. Resumen Final

In [ ]:
print("=" * 70)
print(" " * 20 + "RESUMEN DEL DATASET")
print("=" * 70)

print(f"\n📊 Total de imágenes: {total_count:,}")
print(f"   Train: {train_count:,} ({train_count/total_count*100:.1f}%)")
print(f"   Val:   {val_count:,} ({val_count/total_count*100:.1f}%)")
print(f"   Test:  {test_count:,} ({test_count/total_count*100:.1f}%)")

print(f"\n📦 Total de objetos anotados:")
total_objs = sum(train_objs.values()) + sum(val_objs.values()) + sum(test_objs.values())
print(f"   TOTAL: {total_objs:,}")
print(f"   Train: {sum(train_objs.values()):,}")
print(f"   Val:   {sum(val_objs.values()):,}")
print(f"   Test:  {sum(test_objs.values()):,}")

print(f"\n🏷️  Clases: {len(class_names)}")
for class_id, name in class_names.items():
    print(f"   {class_id}: {name}")

print("\n✅ Dataset listo para entrenamiento")
print("=" * 70)